# 03 · Neural networks: direct vs residual learning

Owner: ML lead.

Same features and architecture as Qiao & Wan (2024). Early stopping on 2019; 2020 untouched until the final evaluation.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "..")
from config import *
from src import models, evaluate, store

SAMPLE = "main"            # "main" = all quoted options, "traded_only" = volume > 0 (see config.SAMPLES)
P = sample_paths(SAMPLE)
P["tables"].mkdir(parents=True, exist_ok=True); P["figures"].mkdir(parents=True, exist_ok=True)
print(SAMPLE, "->", P["pairs"])

In [ ]:
pairs = store.load_pairs(P["pairs"])
pairs = store.load_predictions(pairs, ["hw", "gbm"], P["predictions"])   # from notebook 02
train, val, test = (pairs[pairs.split == s] for s in ("train", "val", "test"))

In [ ]:
nn_resid  = models.NeuralHedger(residual=True).fit(train, val=val)
nn_direct = models.NeuralHedger(residual=False).fit(train, val=val)

In [ ]:
for m in (nn_resid, nn_direct):
    h = pd.DataFrame(m.history_, columns=["epoch", "train", "val"]).set_index("epoch")
    h.plot(title=f"{m.name}: weighted MSE"); plt.show()

In [ ]:
evaluate.add_predictions(pairs, {"nn_resid": nn_resid, "nn_direct": nn_direct}, inplace=True)
MODELS = ["hw", "gbm", "nn_direct", "nn_resid"]
tables = evaluate.standard_report(pairs, MODELS, P["tables"] / "neural")
tables["by_split_type"].round(3)

In [ ]:
tables["crisis"].round(3)

In [ ]:
store.save_predictions(pairs, ["nn_resid", "nn_direct"], P["predictions"])
import torch; torch.save(nn_resid.net_.state_dict(), P["processed"] / "nn_resid.pt")